# AlphaFold 3 inference with pre-computed data pipeline output

*Still experimental/work-in-progress*

This implements
[data pipeline re-use](https://github.com/google-deepmind/alphafold3/blob/main/docs/performance.md#pre-computing-and-reusing-msa-and-templates) with the following addons:
1. Data pipeline output is only ever stored as (gzip-compressed) JSON files.
2. Input sequences are matched to pre-computed data pipeline output **by sequence**. There's no need to track sequences by using a particular set of identifiers (Uniprot/Ensembl/RefSeq) as file names.
3. The pipeline automatically identifies input sequences that do not have pre-computed data pipeline output. It will then run the data pipeline only on the missing sequences, with each missing sequence as a separate SLURM job.

Specify directories (one or more) with pre-computed data pipeline output in `config.yaml`:
```
alphafold3:
  ...
  data_sources: >-
    --data_dir=/cluster/project/beltrao/shared/25.06_alphafold3_msas_yeast
  ...
```

Directories specified with `--data-dir` should contain a file `.af3io_data_index.json` with a pre-generated sequences-to-data JSON mapping. See run_datafill_index.ipynb in this directory and the [datafill.ipynb example in af3io](https://github.com/jurgjn/af3io/).

Run AlphaFold 3 with the following three batch-infer "steps":
- `alphafold3_datafill_missing` checks all input sequences against `data_sources` to find sequences without pre-computed data pipeline output:
    - reads all input JSONs from `alphafold3_jsons/`
    - writes missing sequence input JSONs to `alphafold3_missing/`
- `alphafold3_datafill_msas` runs AlphaFold 3 data pipeline for missing sequences:
    - runs data pipeline for all missing sequence from `alphafold3_missing/`
    - writes the output to `alphafold3_msas/`
- `alphafold3_datafill_predictions` runs theinference step:
    - reads input JSONs from `alphafold3_jsons/`
    - creates data pipeline output (in local scratch) based `data_sources` and/or `alphafold3_msas/`
    - writes structure predictions to `alphafold3_predictions/`

In [1]:
# Run under the virtual environment at .venv/ in the batch-infer directory
# The batch-infer script should be included in the PATH after activating the environment, e.g. `source .venv/bin/activate`
!which batch-infer

/cluster/project/beltrao/jjaenes/26.01_batch-infer_foldx/.venv/bin/batch-infer


In [2]:
# Use human & yeast pre-computed MSAs by specifying data-sources:
!cat config.yaml


alphafold3:
  data_sources: >-
    --data_dir=/cluster/project/beltrao/jjaenes/25.06.03_batch-infer/results/alphafold3_yeast/alphafold3_msas
    --data_dir=/cluster/work/beltrao/jjaenes/25.04.02_batch-infer-projects/af3_human/alphafold3_msas


In [3]:
# Create two input JSON files with a short human protein with a pre-computed MSAs (atox1)
# Add the same protein (atox1) with 2 or 5 glycines added at the end as "missing" chains that do not have a pre-computed MSA
!af3io input-create alphafold3_jsons/atox1_atox1gly5.json \
    --sequence MPKHEFSVDMTCGGCAEAVSRVLNKLGGVKYDIDLPNKKVCIESEHSMDTLLATLKKTGKTVSYLGLE \
    --sequence MPKHEFSVDMTCGGCAEAVSRVLNKLGGVKYDIDLPNKKVCIESEHSMDTLLATLKKTGKTVSYLGLEGGGGG
!af3io input-create alphafold3_jsons/atox1_atox1gly2_atox1gly5.json \
    --sequence MPKHEFSVDMTCGGCAEAVSRVLNKLGGVKYDIDLPNKKVCIESEHSMDTLLATLKKTGKTVSYLGLE \
    --sequence MPKHEFSVDMTCGGCAEAVSRVLNKLGGVKYDIDLPNKKVCIESEHSMDTLLATLKKTGKTVSYLGLEGG \
    --sequence MPKHEFSVDMTCGGCAEAVSRVLNKLGGVKYDIDLPNKKVCIESEHSMDTLLATLKKTGKTVSYLGLEGGGGG
# input JSONs stored under alphafold3_jons/
!ls -l alphafold3_jsons/

Setting name to: atox1_atox1gly5
Write:	/cluster/project/beltrao/jjaenes/26.01_batch-infer_foldx/results/alphafold3_datafill/alphafold3_jsons/atox1_atox1gly5.json
Setting name to: atox1_atox1gly2_atox1gly5
Write:	/cluster/project/beltrao/jjaenes/26.01_batch-infer_foldx/results/alphafold3_datafill/alphafold3_jsons/atox1_atox1gly2_atox1gly5.json
total 8
-rw-r--r-- 1 jjaenes biol-imsb-beltrao 637 Jan 30 10:51 atox1_atox1gly2_atox1gly5.json
-rw-r--r-- 1 jjaenes biol-imsb-beltrao 475 Jan 30 10:51 atox1_atox1gly5.json


In [4]:
# Find missing sequences (./ means use the current directory)
!batch-infer alphafold3_datafill_missing ./ | sbatch

Submitted batch job 55695417


In [5]:
# For every missing sequence (atox1 with 2 or 5 glycines), there's now an input JSON under alphafold3_missing/
# The file names consists of the original input JSON, and the id of the missing sequence
# e.g. atox1_atox1gly2_atox1gly5_b refers to sequence B from alphafold3_jsons/atox1_atox1gly2_atox1gly5.json
# Duplicate missing chains are handled correctly, i.e. atox1gly5 appears in two input JSONs but only gets one "missing" JSON
!ls -l alphafold3_missing/

total 8
-rw-r--r-- 1 jjaenes biol-imsb-beltrao 334 Jan 30 10:58 atox1_atox1gly2_atox1gly5_b.json
-rw-r--r-- 1 jjaenes biol-imsb-beltrao 327 Jan 30 10:58 atox1_atox1gly5_b.json


In [6]:
# Run data pipeline only for the missing sequences
!batch-infer alphafold3_datafill_msas ./ | sbatch

Submitted batch job 55696091


In [8]:
# One job per every missing chain/file under alphafold3_missing/
!squeue --format="%.18i %.12P %.128j %.8T %.16M %.16l %40R" | column -t --table-right 1,5,6 | grep alphafold3_datafill_msas

55697434  normal.120h  batch-infer:alphafold3_datafill_msas=alphafold3_datafill                            RUNNING        7:51  7-00:00:00  eu-a2p-534             
55697634  normal.4h    alphafold3_datafill_msas_run:id=atox1_atox1gly5_b                                   RUNNING        6:14     4:00:00  eu-a2p-360             
55697636  normal.4h    alphafold3_datafill_msas_run:id=atox1_atox1gly2_atox1gly5_b                         RUNNING        6:14     4:00:00  eu-a2p-370             


In [1]:
# Data pipeline output for missing sequences stored under alphafold3_msas/
!ls -l alphafold3_msas/

total 772
-rw-r--r-- 1 jjaenes biol-imsb-beltrao 423232 Jan 30 11:49 atox1_atox1gly2_atox1gly5_b_data.json.gz
-rw-r--r-- 1 jjaenes biol-imsb-beltrao 353389 Jan 30 11:41 atox1_atox1gly5_b_data.json.gz


In [2]:
# Run predictions, creating temporary data pipeline output on local scratch as-needed
!batch-infer alphafold3_datafill_predictions ./ | sbatch

Submitted batch job 55699713


In [1]:
# Predictions stored as zip-compressed archives under alphafold3_predictions/
!ls -l alphafold3_predictions/

total 1000
-rw-r--r-- 1 jjaenes biol-imsb-beltrao 645377 Jan 30 23:27 atox1_atox1gly2_atox1gly5.zip
-rw-r--r-- 1 jjaenes biol-imsb-beltrao 366419 Jan 30 23:28 atox1_atox1gly5.zip


In [ ]:
# Cleanup
#!rm -rf .snakemake/
#!rm -rf .snakemake-eu/
#!rm -rf alphafold3_jsons
#!rm -rf alphafold3_missing
#!rm -rf alphafold3_msas
#!rm -rf alphafold3_predictions